# 03 — Flips: which answers changed after quantization?

Compares each quantized model with the original FP16 model **question by question**.
No GPU needed (set Accelerator to **None** to save GPU hours).

**What it computes**
1. Accuracy of every model (with 95% confidence intervals)
2. **Flips** against FP16: *harmful* (FP16 right → quantized wrong) and *repair* (wrong → right), with a McNemar test  → **RQ1**
3. **RTN vs GPTQ head to head** on the questions FP16 got right (McNemar), and how much their broken questions overlap
4. **Rule-based first labels** for each harmful flip (degeneration, no answer); the rest go to the LLM judge in notebook 04 → **RQ2**
5. Does FP16's **confidence** predict which answers flip? (AUROC) → **RQ3**
6. Do answers get **longer** after quantization? (token counts)

**Input:** the `gen_<TAG>_gsm8k.jsonl` and `gen_<TAG>_triviaqa.jsonl` files from notebook 01.
Easiest way: download them from each 01 run's **Output** tab, then **Create → New Dataset**, upload them all,
and attach that dataset here (**+ Add Input → Your Datasets**). When more runs finish, add their files with
**New Version** of the same dataset. The notebook finds the files wherever they are under `/kaggle/input`.

**Saved files:** `03_results.json` (all numbers), `03_harmful_flips.jsonl` (every harmful flip with both outputs,
for the judge), `03_per_question.csv` (one row per question, one column per model).

In [ ]:
# 1. Settings
SIZE      = "1.5b"     # model size of the answer files to use: "0.5b", "1.5b" or "3b"
TQA_SCORE = "lenient"  # TriviaQA: "lenient" = also accept long correct answers (main), "strict" = 01's rule
REFERENCE = "fp16"
ORDER     = ["fp16", "rtn4", "gptq4", "gptq4math"]      # display order (missing models are skipped)
NAMES     = {"fp16": "FP16", "rtn4": "RTN-4", "gptq4": "GPTQ-4 (C4, act-order)", "gptq4math": "GPTQ-4 (math)"}
TASKS     = ["gsm8k", "triviaqa"]
N_BOOT    = 2000      # bootstrap samples for confidence intervals
SEED      = 0
OUT_DIR   = "/kaggle/working"
SUF       = "" if SIZE == "1.5b" else f"_{SIZE}"     # output names: 03_results.json (1.5b), 03_results_3b.json, ...

In [ ]:
# 2. Find and load the answer files
import glob, json, os, re
import numpy as np, pandas as pd

def read_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return rows

found = {}
# file names: gen_fp16_gsm8k.jsonl (1.5b study) or gen_3b-fp16_gsm8k.jsonl (other sizes); .txt copies also work
for p in sorted(glob.glob("/kaggle/input/**/gen_*_*", recursive=True) + glob.glob(f"{OUT_DIR}/gen_*_*")):
    m = re.match(r"gen_(.+?)_(gsm8k|triviaqa)(\.jsonl)?(\.txt)?$", os.path.basename(p))
    if not m:
        continue
    tag, task = m.group(1), m.group(2)
    if SIZE == "1.5b":
        if "-" in tag:                       # another size's file
            continue
    elif tag.startswith(f"{SIZE}-"):
        tag = tag[len(SIZE) + 1:]
    else:
        continue
    if (tag, task) not in found:
        found[(tag, task)] = p

data = {}   # data[task][tag] = DataFrame indexed by question id
for (tag, task), p in sorted(found.items()):
    df = pd.DataFrame(read_jsonl(p))
    df["id"] = df["id"].astype(str)
    df = df.drop_duplicates("id").set_index("id")
    data.setdefault(task, {})[tag] = df
    print(f"{task:9s} {tag:10s} {len(df):6d} answers   <- {p}")

for task in TASKS:
    assert REFERENCE in data.get(task, {}), f"FP16 file for {task} not found: attach gen_fp16_{task}.jsonl"
MODELS = [t for t in ORDER if all(t in data[task] for task in TASKS)]
QUANT  = [t for t in MODELS if t != REFERENCE]
print("\nmodels with both tasks:", MODELS)

# keep only the questions every model answered, in the same order
for task in TASKS:
    common = sorted(set.intersection(*[set(data[task][t].index) for t in MODELS]))
    for t in MODELS:
        data[task][t] = data[task][t].loc[common]
    print(f"{task}: {len(common)} questions answered by every model")

In [ ]:
# 2b. Re-score TriviaQA: also accept correct answers that are long or wordy
#   strict  = the original rule from 01 (answer inside a short first line, <= 10 words)
#   lenient = an accepted answer appears as whole words anywhere in the first line, any length
import unicodedata, string

def normalize_answer(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(ch for ch in s if not unicodedata.combining(ch)).lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def has_alias(text, aliases):
    p = normalize_answer(text)
    return any(a and re.search(r"(^| )" + re.escape(a) + r"( |$)", p) for a in aliases)

if TQA_SCORE == "lenient":
    from datasets import load_dataset
    tqa = load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation")
    ALIASES = {str(x["question_id"]): {normalize_answer(a) for a in x["answer"]["aliases"] + [x["answer"]["value"]] if a.strip()}
               for x in tqa}
    for t, df in data["triviaqa"].items():
        df["correct_strict"] = df["correct"].astype(bool)
        first = df["output"].fillna("").map(lambda o: o.strip().split("\n")[0])
        df["correct"] = [bool(c) or has_alias(f, ALIASES.get(i, set()))
                         for i, f, c in zip(df.index, first, df["correct_strict"])]
        print(f"{t:8s} strict {100*df['correct_strict'].mean():.1f}%  ->  lenient {100*df['correct'].mean():.1f}%"
              f"   (+{int((df['correct'] & ~df['correct_strict']).sum())} answers now correct)")

In [ ]:
# 3. Accuracy of every model (95% bootstrap confidence interval)
rng = np.random.default_rng(SEED)

def boot_ci(x, n=N_BOOT):
    x = np.asarray(x, dtype=float)
    idx = rng.integers(0, len(x), size=(n, len(x)))
    m = x[idx].mean(axis=1)
    return np.percentile(m, 2.5), np.percentile(m, 97.5)

results = {"accuracy": {}, "flips": {}, "head_to_head": {}, "auroc": {}, "length": {}, "rule_labels": {}}
rows = []
for task in TASKS:
    for t in MODELS:
        c = data[task][t]["correct"].astype(bool).values
        lo, hi = boot_ci(c)
        results["accuracy"][f"{t}_{task}"] = {"acc": c.mean(), "ci95": [lo, hi], "n": len(c)}
        rows.append({"task": task, "model": NAMES[t], "accuracy %": 100 * c.mean(),
                     "95% CI": f"{100*lo:.1f} – {100*hi:.1f}", "correct": int(c.sum()), "n": len(c)})
pd.DataFrame(rows).round(2)

In [ ]:
# 4. RQ1 — flips against FP16 (McNemar exact test on the changed answers)
from scipy.stats import binomtest

def mcnemar(b, c):
    """Exact McNemar: b and c are the two kinds of disagreement."""
    return 1.0 if b + c == 0 else binomtest(min(b, c), b + c, 0.5).pvalue

rows = []
for task in TASKS:
    ref = data[task][REFERENCE]["correct"].astype(bool).values
    for t in QUANT:
        q = data[task][t]["correct"].astype(bool).values
        harmful = int((ref & ~q).sum()); repair = int((~ref & q).sum())
        r = {"harmful": harmful, "repair": repair,
             "stable_right": int((ref & q).sum()), "stable_wrong": int((~ref & ~q).sum()),
             "flip_rate": (harmful + repair) / len(ref),
             "harmful_rate_of_fp16_right": harmful / max(ref.sum(), 1),
             "net_change": (repair - harmful) / len(ref),
             "mcnemar_p": mcnemar(harmful, repair)}
        results["flips"][f"{t}_{task}"] = r
        rows.append({"task": task, "model": NAMES[t], "harmful": harmful, "repair": repair,
                     "all flips %": 100 * r["flip_rate"],
                     "harmful % of FP16-right": 100 * r["harmful_rate_of_fp16_right"],
                     "accuracy change (points)": 100 * r["net_change"], "McNemar p": r["mcnemar_p"]})
flip_table = pd.DataFrame(rows)
flip_table.round(4)

**How to read it:** *harmful* = FP16 right, quantized wrong; *repair* = the reverse. Accuracy change is
(repair − harmful) / n, so a small accuracy change can hide many flips in both directions.
McNemar p < 0.05 means the model really is worse (or better) than FP16, not just noise.

In [ ]:
# 5. RQ1 — RTN vs GPTQ head to head, only on questions FP16 got right
from itertools import combinations
rows = []
for task in TASKS:
    ref = data[task][REFERENCE]["correct"].astype(bool).values
    for a, b in combinations(QUANT, 2):
        A = ~data[task][a]["correct"].astype(bool).values & ref      # broken by model a
        B = ~data[task][b]["correct"].astype(bool).values & ref      # broken by model b
        only_a, only_b, both = int((A & ~B).sum()), int((~A & B).sum()), int((A & B).sum())
        jacc = both / max(int((A | B).sum()), 1)
        r = {"broken_by_both": both, f"only_{a}": only_a, f"only_{b}": only_b,
             "overlap_jaccard": jacc, "mcnemar_p": mcnemar(only_a, only_b)}
        results["head_to_head"][f"{a}_vs_{b}_{task}"] = r
        rows.append({"task": task, "pair": f"{NAMES[a]} vs {NAMES[b]}", "broken by both": both,
                     f"only first": only_a, "only second": only_b,
                     "overlap (Jaccard)": jacc, "McNemar p": r["mcnemar_p"]})
pd.DataFrame(rows).round(4) if rows else print("needs at least two quantized models")

**Overlap (Jaccard)** = questions broken by both ÷ questions broken by either. Near 1: both methods break the
*same* questions. Low: they fail on *different* questions, even if their accuracy is similar.

In [ ]:
# 6. RQ2 (first pass) — rule-based labels for each harmful flip; everything else goes to the LLM judge (notebook 04)
REFUSAL = re.compile(r"\b(i don'?t know|i do not know|not sure|cannot (answer|determine)|unknown|no answer|"
                     r"i'?m sorry|unable to)\b", re.I)

def normalize(s):
    s = re.sub(r"[^a-z0-9 ]", " ", str(s).lower())
    return " ".join(w for w in s.split() if w not in {"a", "an", "the"})

def repetitive(text, n=6, min_words=60, threshold=0.6):
    """Many repeated 6-word phrases = the model is looping."""
    w = str(text).split()
    if len(w) < min_words:
        return False
    grams = [tuple(w[i:i + n]) for i in range(len(w) - n + 1)]
    return len(set(grams)) / len(grams) < threshold

def rule_label(task, q_row, question):
    out = q_row["output"]
    if task == "gsm8k":
        if q_row["hit_limit"] or repetitive(out):
            return "degeneration"              # looping or cut off at 1,024 tokens
        if q_row["extracted_by"] != "boxed":
            return "degeneration"              # no final boxed answer
        return "judge"                         # conceptual / method / execution / reasoning
    pred = normalize(q_row["pred"])
    if not pred or repetitive(out, n=3, min_words=12):
        return "degeneration"
    if REFUSAL.search(str(q_row["pred"])) or (len(pred.split()) >= 3 and pred in normalize(question)):
        return "no_answer"                     # refuses, or repeats the question
    return "judge"                             # near-miss / unrelated

flip_records = []
for task in TASKS:
    ref = data[task][REFERENCE]
    for t in QUANT:
        q = data[task][t]
        mask = ref["correct"].astype(bool) & ~q["correct"].astype(bool)
        for qid in ref.index[mask.values]:
            r0, r1 = ref.loc[qid], q.loc[qid]
            flip_records.append({
                "task": task, "model": t, "id": qid, "question": r0["question"], "gold": r0["gold"],
                "fp16_output": r0["output"], "fp16_pred": r0["pred"],
                "quant_output": r1["output"], "quant_pred": r1["pred"],
                "fp16_tokens": int(r0["n_tokens"]), "quant_tokens": int(r1["n_tokens"]),
                "quant_hit_limit": bool(r1["hit_limit"]),
                "fp16_mean_logprob": float(r0["mean_logprob"]), "fp16_min_logprob": float(r0["min_logprob"]),
                "rule_label": rule_label(task, r1, r0["question"])})

flips = pd.DataFrame(flip_records)
with open(f"{OUT_DIR}/03_harmful_flips{SUF}.jsonl", "w") as f:
    for r in flip_records:
        f.write(json.dumps(r) + "\n")
print(len(flips), f"harmful flips saved to 03_harmful_flips{SUF}.jsonl\n")

if len(flips):
    tab = pd.crosstab([flips["task"], flips["model"]], flips["rule_label"], margins=True)
    for (task, t), g in flips.groupby(["task", "model"]):
        results["rule_labels"][f"{t}_{task}"] = g["rule_label"].value_counts().to_dict()
    display(tab)

Only the clear-cut cases are labelled by rules (loops, cut-off answers, no final answer, refusals). The rest
(`judge`) get one of the content labels from the LLM judge in notebook 04: *conceptual, method, execution,
reasoning* for math and *near-miss, unrelated* for facts. The χ² test for RQ2 is run after that.

In [ ]:
# 7. RQ3 — does FP16's confidence predict which of its right answers will flip?
from sklearn.metrics import roc_auc_score

def boot_auc(y, s, n=N_BOOT):
    y, s = np.asarray(y), np.asarray(s)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max():
            vals.append(roc_auc_score(y[i], s[i]))
    return np.percentile(vals, 2.5), np.percentile(vals, 97.5)

rows = []
for task in TASKS:
    ref = data[task][REFERENCE]
    right = ref["correct"].astype(bool).values
    for t in QUANT:
        y = (~data[task][t]["correct"].astype(bool).values)[right].astype(int)   # 1 = this right answer flipped
        if y.min() == y.max():
            continue
        for feat in ["mean_logprob", "min_logprob"]:
            s = -ref[feat].values[right]          # lower confidence -> higher flip score
            auc = roc_auc_score(y, s); lo, hi = boot_auc(y, s)
            results["auroc"][f"{t}_{task}_{feat}"] = {"auroc": auc, "ci95": [lo, hi], "n": int(len(y)), "flips": int(y.sum())}
            rows.append({"task": task, "model": NAMES[t], "confidence": feat, "AUROC": auc,
                         "95% CI": f"{lo:.3f} – {hi:.3f}", "flips / FP16-right": f"{int(y.sum())} / {len(y)}"})
pd.DataFrame(rows).round(3)

**AUROC** = chance that a randomly chosen *flipped* answer had lower FP16 confidence than a randomly chosen
*stable* one. 0.5 = confidence tells us nothing; 1.0 = perfect warning signal.

In [ ]:
# 8. Do answers get longer after quantization? (median tokens on the same questions)
rows = []
for task in TASKS:
    ref_tok = data[task][REFERENCE]["n_tokens"].astype(float).values
    for t in QUANT:
        q_tok = data[task][t]["n_tokens"].astype(float).values
        hit = int(data[task][t]["hit_limit"].astype(bool).sum())
        r = {"median_fp16": float(np.median(ref_tok)), "median_quant": float(np.median(q_tok)),
             "mean_ratio": float(q_tok.sum() / ref_tok.sum()),
             "hit_limit_fp16": int(data[task][REFERENCE]["hit_limit"].astype(bool).sum()), "hit_limit_quant": hit}
        results["length"][f"{t}_{task}"] = r
        rows.append({"task": task, "model": NAMES[t], "median tokens FP16": r["median_fp16"],
                     "median tokens quant": r["median_quant"], "total tokens vs FP16": f"{100*(r['mean_ratio']-1):+.1f}%",
                     "hit cap (FP16 → quant)": f"{r['hit_limit_fp16']} → {hit}"})
pd.DataFrame(rows)

In [ ]:
# 9. A few harmful flips side by side (first 3 per model and task)
def short(s, n=400):
    s = " ".join(str(s).split())
    return s if len(s) <= n else s[:n] + " …"

for (task, t), g in flips.groupby(["task", "model"]) if len(flips) else []:
    print("=" * 100, f"\n{task} · {NAMES[t]} · {len(g)} harmful flips")
    for _, r in g.head(3).iterrows():
        print(f"\nQ: {short(r['question'], 250)}\n gold: {r['gold']} | rule label: {r['rule_label']}")
        print(f" FP16  ({r['fp16_tokens']} tok) -> {r['fp16_pred']}: {short(r['fp16_output'])}")
        print(f" QUANT ({r['quant_tokens']} tok) -> {r['quant_pred']}: {short(r['quant_output'])}")

In [ ]:
# 10. Save everything
wide = []
for task in TASKS:
    w = pd.DataFrame({t: data[task][t]["correct"].astype(bool) for t in MODELS})
    w.insert(0, "task", task)
    wide.append(w)
pd.concat(wide).to_csv(f"{OUT_DIR}/03_per_question{SUF}.csv")

def clean(o):
    if isinstance(o, dict):  return {k: clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [clean(v) for v in o]
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    return o

results["models"] = MODELS
with open(f"{OUT_DIR}/03_results{SUF}.json", "w") as f:
    json.dump(clean(results), f, indent=1)
print(f"saved: 03_results{SUF}.json, 03_harmful_flips{SUF}.jsonl, 03_per_question{SUF}.csv")